In [40]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as Sum, month, dense_rank
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 03:59:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = spark.read\
.option("header", "true")\
.option("inferSchema","false")\
.format("csv")\
.load("files/orders.txt")

In [5]:
df.show()

+--------+-------+----------+-----+
|order_id|item_id|sales_date|price|
+--------+-------+----------+-----+
|     101|      1|2022-06-03| 5.99|
|     102|      2|2022-06-03| 4.99|
|     103|      1|2022-06-04| 5.99|
|     104|      3|2022-06-04| 3.99|
|     105|      2|2022-07-01| 4.99|
|     106|      3|2022-07-02| 3.99|
|     107|      1|2022-07-02| 5.99|
+--------+-------+----------+-----+



In [6]:
df.createOrReplaceTempView("orders")

In [54]:
spark.sql(
    """
    with cte as (
    select month(sales_date) as month,
    item_id,
    sum(price) as total
    from orders
    group by month(sales_date), item_id
    )
    select month, item_id, total,
    dense_rank() over(partition by month order by total desc) as item_rank
    from cte
    """
).show()

+-----+-------+-----+---------+
|month|item_id|total|item_rank|
+-----+-------+-----+---------+
|    6|      1|11.98|        1|
|    6|      2| 4.99|        2|
|    6|      3| 3.99|        3|
|    7|      1| 5.99|        1|
|    7|      2| 4.99|        2|
|    7|      3| 3.99|        3|
+-----+-------+-----+---------+



In [70]:
df = spark.read\
    .option("header", "true")\
    .csv("files/employees.txt")

In [71]:
df.show()

+-----------+----------------+------+-------------+----------+
|employee_id|            name|salary|department_id|manager_id|
+-----------+----------------+------+-------------+----------+
|          1|   Emma Thompson|  3800|            1|         2|
|          2|Daniel Rodriguez|  2230|            1|        10|
|          3|    Olivia Smith|  8000|            1|         8|
|          4|    Noah Johnson|  6800|            2|         8|
|          5| Sophia Martinez|  1750|            1|        10|
|          8|   William Davis|  7000|            2|      null|
|         10|  James Anderson|  4000|            1|      null|
+-----------+----------------+------+-------------+----------+



In [72]:
df.createOrReplaceTempView("employees")

In [88]:
spark.sql(
    """
    select e.employee_id as empId, e.name from 
    employees e inner join employees m on e.manager_id = m.employee_id 
    where e.salary>m.salary
    """
).show()

+-----+------------+
|empId|        name|
+-----+------------+
|    3|Olivia Smith|
+-----+------------+



In [91]:
burgers = spark.read.csv("files/burgers.txt", header=True)

In [92]:
burgers.show()

+---------+---------------+-----+
|burger_id|    burger_type|price|
+---------+---------------+-----+
|        1|        Big Mac| 3.99|
|        2|Quarter Pounder| 3.79|
|        3|   Cheeseburger| 1.69|
+---------+---------------+-----+



In [93]:
sales = spark.read.csv("files/sales.txt", header=True)

In [94]:
sales.show()

+-------+---------+---------+--------+
|sale_id|burger_id|branch_id|quantity|
+-------+---------+---------+--------+
|      1|        1|        1|     100|
|      2|        1|        2|     150|
|      3|        2|        1|      80|
+-------+---------+---------+--------+



In [95]:
branch = spark.read.csv("files/branch.txt", header=True)

In [97]:
branch.show()

+---------+--------+
|branch_id|    city|
+---------+--------+
|        1|New York|
|        2| Chicago|
+---------+--------+



In [98]:
burgers.createOrReplaceTempView("burgers")
sales.createOrReplaceTempView("sales")
branch.createOrReplaceTempView("branch")

In [112]:
spark.sql(
    """
    select b.burger_type,
    br.city,
    (count(s.burger_id)) as counts
    from burgers b
    inner join sales s on b.burger_id = s.burger_id
    inner join branch br on s.branch_id = br.branch_id
    group by 1,2
    """
).show()

+---------------+--------+------+
|    burger_type|    city|counts|
+---------------+--------+------+
|Quarter Pounder|New York|     1|
|        Big Mac|New York|     1|
|        Big Mac| Chicago|     1|
+---------------+--------+------+



In [115]:
orders=spark.read.csv("files/orders_mcd.txt", header=True)

In [116]:
orders.show()

+--------+-----------+-------+----------+
|order_id|customer_id|meal_id|order_date|
+--------+-----------+-------+----------+
|    1001|        201|    301|2022-02-10|
|    1002|        202|    302|2022-02-15|
|    1003|        201|    301|2022-02-18|
|    1004|        203|    302|2022-03-01|
|    1005|        204|    303|2022-03-02|
|    1006|        201|    303|2022-03-05|
|    1007|        204|    303|2022-03-10|
+--------+-----------+-------+----------+



In [119]:
customers=spark.read.csv("files/customers_mcd.txt", header=True)

In [120]:
customers.show()

+-----------+--------------+
|customer_id|    preference|
+-----------+--------------+
|        201|Non-vegetarian|
|        202|    Vegetarian|
|        203|Non-vegetarian|
|        204|Non-vegetarian|
+-----------+--------------+



In [121]:
orders.createOrReplaceTempView("orders_mcd")
customers.createOrReplaceTempView("customers_mcd")

In [137]:
spark.sql(
    """
    select c.customer_id, count(o.order_id) as orders, c.preference
    from orders_mcd o inner join customers_mcd c on o.customer_id=c.customer_id
    where o.order_date between '2022-02-01' and '2022-02-28'
    and c.preference='Non-vegetarian'
    group by 1,3
    having count(o.order_id)>=2
    
    """
).show()

+-----------+------+--------------+
|customer_id|orders|    preference|
+-----------+------+--------------+
|        201|     2|Non-vegetarian|
+-----------+------+--------------+



In [138]:
daily_sales = spark.read.csv("files/daily_sales.txt", header=True)

In [139]:
daily_sales.show()

+----------+-------+--------+
|      date|item_id|quantity|
+----------+-------+--------+
|2022-08-01|      1|     300|
|2022-08-01|      2|     150|
|2022-08-02|      1|     400|
|2022-08-02|      2|     200|
|2022-08-02|      3|     100|
|2022-08-03|      1|     350|
|2022-08-03|      3|     120|
+----------+-------+--------+



In [140]:
menu = spark.read.csv("files/menu.txt", header=True)

In [141]:
menu.show()

+-------+-----------------+
|item_id|        item_name|
+-------+-----------------+
|      1|          Big Mac|
|      2|  Quarter Pounder|
|      3|Chicken McNuggets|
+-------+-----------------+



In [142]:
daily_sales.createOrReplaceTempView("daily_sales")
menu.createOrReplaceTempView("menu")

In [146]:
spark.sql(
    """
    select m.item_name, avg(ds.quantity) avg_daily_sales
    from menu m
    inner join daily_sales ds on ds.item_id=m.item_id
    group by 1
    """
).show()

+-----------------+---------------+
|        item_name|avg_daily_sales|
+-----------------+---------------+
|Chicken McNuggets|          110.0|
|  Quarter Pounder|          175.0|
|          Big Mac|          350.0|
+-----------------+---------------+

